# Layer-metrics comparison

Loads all `layer_metrics_*.csv` files produced by `run_experiments.py`, filters to a chosen experiment config, and plots algorithm-vs-algorithm comparisons faceted by layer type (`q_proj`, `k_proj`, `v_proj`, `o_proj`, `gate_proj`, `up_proj`, `down_proj`).

Key metric: **`rel_error`** — the fraction of Wanda mass `∑ (W·‖x‖)²` thrown away by pruning. Lower is better. This is the right metric for comparing pruning algorithms because (a) it's the quantity Wanda scores were designed to minimize, and (b) it's normalized so layers of different sizes are comparable.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from IPython.display import display

OUTPUT_FOLDER = "experiment_results"

LAYER_TYPES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
LAYER_GROUP = {lt: ("Attention" if lt in {"q_proj", "k_proj", "v_proj", "o_proj"} else "MLP")
               for lt in LAYER_TYPES}

# Consistent color per algorithm across all plots
ALG_COLORS = {
    "our_tetris":                 "#d62728",
    "original_tetris":            "#1f77b4",
    "sort_columns_by_norm":       "#2ca02c",
    "block_wanda":                "#ff7f0e",
    "random_permutation_pruning": "#9467bd",
    "random_swaps_find_mask":     "#8c564b",
}
def alg_color(alg):
    return ALG_COLORS.get(alg, "#555555")

## 1. Load all CSVs

In [ ]:
csv_files = sorted(glob.glob(os.path.join(OUTPUT_FOLDER, "layer_metrics_*.csv")))
print(f"Found {len(csv_files)} CSVs:")
for f in csv_files:
    print(f"  {os.path.basename(f)}")

dfs = [pd.read_csv(f) for f in csv_files]
df = pd.concat(dfs, ignore_index=True)

# Derived columns
df["retained_frac"] = df["retained_wanda_mass"] / df["total_wanda_mass"]
df["block_size"] = df["block_rows"].astype(str) + "x" + df["block_cols"].astype(str)
df["layer_group"] = df["layer_type"].map(LAYER_GROUP)

print(f"\nTotal rows: {len(df)}")
print(f"Algorithms:  {sorted(df.algorithm.unique())}")
print(f"Block sizes: {sorted(df.block_size.unique())}")
print(f"Sparsities:  {sorted(df.sparsity.unique())}")

## 2. Pick an experiment config to compare

Each CSV is one algorithm at one `(block_size, sparsity, noise, iters, swaps)` setting. To get a clean comparison plot, fix everything except `algorithm`.

In [ ]:
# Filter knobs — edit to pick which experiment slice to compare
FILTER_BLOCK_ROWS = 1
FILTER_BLOCK_COLS = 2
FILTER_SPARSITY   = 0.5

mask = (
    (df.block_rows == FILTER_BLOCK_ROWS) &
    (df.block_cols == FILTER_BLOCK_COLS) &
    (df.sparsity   == FILTER_SPARSITY)
)
df_exp = df[mask].copy()
print(f"Filtered rows: {len(df_exp)}")
print(f"Algorithms in filter: {sorted(df_exp.algorithm.unique())}")

# Baseline to compute deltas against — change if you want a different reference
BASELINE_ALG = "original_tetris"
assert BASELINE_ALG in df_exp.algorithm.unique(), \
    f"Baseline '{BASELINE_ALG}' not found. Available: {sorted(df_exp.algorithm.unique())}"

## 3. Summary tables

One-line-per-algorithm summary, and a per-layer-type breakdown.

In [ ]:
# Overall per-algorithm summary
overall = (df_exp.groupby("algorithm")
           .agg(mean_rel_error=("rel_error", "mean"),
                median_rel_error=("rel_error", "median"),
                mean_retained_frac=("retained_frac", "mean"),
                mean_time_sec=("total_time_sec", "mean"),
                total_time_sec=("total_time_sec", "sum"),
                n_layers=("rel_error", "count"))
           .sort_values("mean_rel_error"))
print("=== Overall (all layers pooled) ===")
display(overall.round(5))

In [ ]:
# Per-algorithm × per-layer-type: mean rel_error
by_type = (df_exp.groupby(["layer_type", "algorithm"])["rel_error"]
           .mean().unstack("algorithm"))
# Keep layer_type order consistent
by_type = by_type.reindex([lt for lt in LAYER_TYPES if lt in by_type.index])
print("=== Mean rel_error per layer type × algorithm (lower is better) ===")
display(by_type.round(5))

# Highlight best algorithm per layer type
def _highlight_min(row):
    is_min = row == row.min()
    return ["font-weight: bold; background-color: #d4f4dd" if v else "" for v in is_min]
display(by_type.round(5).style.apply(_highlight_min, axis=1))

## 4. Main plot — rel_error per layer, faceted by layer type

One subplot per layer type. x-axis is the transformer block index (0..N-1). One line per algorithm.

In [ ]:
def faceted_lineplot(data, metric, ylabel, title, baseline_alg=None, logy=False):
    """2x4 grid of subplots; row 1 = attention projections, row 2 = MLP projections."""
    fig, axes = plt.subplots(2, 4, figsize=(18, 8), sharex=True)
    axes_flat = axes.flatten()

    algorithms = sorted(data.algorithm.unique())

    for i, lt in enumerate(LAYER_TYPES):
        ax = axes_flat[i]
        sub = data[data.layer_type == lt].sort_values("layer_idx")

        for alg in algorithms:
            ad = sub[sub.algorithm == alg]
            if len(ad) == 0:
                continue
            lw = 2.2 if alg == baseline_alg else 1.4
            ls = "--" if alg == baseline_alg else "-"
            ax.plot(ad["layer_idx"], ad[metric],
                    label=alg, color=alg_color(alg),
                    linewidth=lw, linestyle=ls,
                    marker="o", markersize=3.5, alpha=0.9)

        ax.set_title(lt, fontsize=12, fontweight="bold")
        ax.grid(True, linestyle="--", alpha=0.35)
        if logy:
            ax.set_yscale("log")
        if i % 4 == 0:
            ax.set_ylabel(ylabel, fontsize=11)
        if i >= 4:
            ax.set_xlabel("Transformer block index", fontsize=10)

    # Hide unused subplot, put legend there
    axes_flat[7].axis("off")
    handles = [Line2D([0], [0], color=alg_color(a), lw=2.0,
                      linestyle="--" if a == baseline_alg else "-",
                      marker="o", markersize=4, label=a)
               for a in algorithms]
    axes_flat[7].legend(handles=handles, loc="center", fontsize=11, frameon=False,
                        title="Algorithm", title_fontsize=12)

    fig.suptitle(title, fontsize=14, fontweight="bold", y=1.00)
    fig.tight_layout()
    return fig

fig = faceted_lineplot(
    df_exp,
    metric="rel_error",
    ylabel="Rel. error  (∥ΔW∥²·‖x‖² / ∥W∥²·‖x‖²)",
    title=f"Wanda-weighted relative error per layer — block {FILTER_BLOCK_ROWS}×{FILTER_BLOCK_COLS}, sparsity {FILTER_SPARSITY}",
    baseline_alg=BASELINE_ALG,
)
plt.show()

## 5. Delta-vs-baseline plot

For each algorithm, show `rel_error − rel_error(baseline)` per layer. Points below zero = better than baseline. This is usually the most legible view of *does my algorithm actually improve on the baseline, and in which layers*.

In [ ]:
def delta_vs_baseline(data, baseline_alg, metric="rel_error"):
    # Pivot so we can subtract baseline column
    piv = data.pivot_table(
        index=["layer_type", "layer_idx"],
        columns="algorithm",
        values=metric,
        aggfunc="mean",
    )
    if baseline_alg not in piv.columns:
        raise ValueError(f"Baseline {baseline_alg} not in data")
    delta = piv.subtract(piv[baseline_alg], axis=0).drop(columns=[baseline_alg])
    delta = delta.reset_index()
    return delta

delta_df = delta_vs_baseline(df_exp, BASELINE_ALG, metric="rel_error")

fig, axes = plt.subplots(2, 4, figsize=(18, 8), sharex=True)
axes_flat = axes.flatten()

non_baseline_algs = [a for a in sorted(df_exp.algorithm.unique()) if a != BASELINE_ALG]

for i, lt in enumerate(LAYER_TYPES):
    ax = axes_flat[i]
    sub = delta_df[delta_df.layer_type == lt].sort_values("layer_idx")
    for alg in non_baseline_algs:
        if alg not in sub.columns:
            continue
        ax.plot(sub["layer_idx"], sub[alg],
                label=alg, color=alg_color(alg),
                linewidth=1.4, marker="o", markersize=3.5, alpha=0.9)
    ax.axhline(0, color="black", linewidth=1.3, linestyle="-", alpha=0.7)
    ax.set_title(lt, fontsize=12, fontweight="bold")
    ax.grid(True, linestyle="--", alpha=0.35)
    if i % 4 == 0:
        ax.set_ylabel(f"Δ rel_error vs {BASELINE_ALG}\n(negative = better)", fontsize=10)
    if i >= 4:
        ax.set_xlabel("Transformer block index", fontsize=10)

axes_flat[7].axis("off")
handles = [Line2D([0], [0], color=alg_color(a), lw=1.8, marker="o", markersize=4, label=a)
           for a in non_baseline_algs]
handles.append(Line2D([0], [0], color="black", lw=1.3, label=f"{BASELINE_ALG} (baseline)"))
axes_flat[7].legend(handles=handles, loc="center", fontsize=11, frameon=False,
                    title="Algorithm", title_fontsize=12)

fig.suptitle(
    f"Δ rel_error vs {BASELINE_ALG}  —  block {FILTER_BLOCK_ROWS}×{FILTER_BLOCK_COLS}, sparsity {FILTER_SPARSITY}",
    fontsize=14, fontweight="bold", y=1.00,
)
fig.tight_layout()
plt.show()

## 6. Aggregate view — mean Δ per layer type

Bar chart: one group per layer type, one bar per algorithm. Quickest way to see *on average, is my algorithm better, and in which layer types*.

In [ ]:
agg = (delta_df.melt(id_vars=["layer_type", "layer_idx"],
                     var_name="algorithm", value_name="delta_rel_error")
       .groupby(["layer_type", "algorithm"])["delta_rel_error"]
       .mean().unstack("algorithm")
       .reindex([lt for lt in LAYER_TYPES if lt in delta_df.layer_type.unique()]))

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(agg.index))
width = 0.8 / max(len(agg.columns), 1)

for k, alg in enumerate(agg.columns):
    ax.bar(x + k * width, agg[alg].values, width, label=alg, color=alg_color(alg), alpha=0.9)

ax.axhline(0, color="black", linewidth=1.0)
ax.set_xticks(x + width * (len(agg.columns) - 1) / 2)
ax.set_xticklabels(agg.index)
ax.set_ylabel(f"Mean Δ rel_error vs {BASELINE_ALG}\n(negative = better)")
ax.set_title(f"Mean Δ rel_error per layer type  (block {FILTER_BLOCK_ROWS}×{FILTER_BLOCK_COLS}, sparsity {FILTER_SPARSITY})",
             fontsize=13, fontweight="bold")
ax.grid(True, axis="y", linestyle="--", alpha=0.35)
ax.legend(loc="best", frameon=False)
fig.tight_layout()
plt.show()

## 7. Time per layer, faceted

How expensive is each algorithm, layer by layer.

In [ ]:
fig = faceted_lineplot(
    df_exp,
    metric="total_time_sec",
    ylabel="Pruning time (s)",
    title=f"Per-layer pruning time — block {FILTER_BLOCK_ROWS}×{FILTER_BLOCK_COLS}, sparsity {FILTER_SPARSITY}",
    baseline_alg=BASELINE_ALG,
    logy=True,
)
plt.show()

## 8. Quality–cost tradeoff (Pareto view)

Each point is one `(algorithm, layer)` pair. x = time, y = rel_error. A point in the lower-left corner is both fast and accurate.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for alg in sorted(df_exp.algorithm.unique()):
    ad = df_exp[df_exp.algorithm == alg]
    ax.scatter(ad["total_time_sec"], ad["rel_error"],
               label=alg, color=alg_color(alg), alpha=0.6, s=35, edgecolor="white", linewidth=0.5)

ax.set_xscale("log")
ax.set_xlabel("Pruning time per layer (s, log scale)")
ax.set_ylabel("Rel. error (lower is better)")
ax.set_title(f"Quality–cost scatter — block {FILTER_BLOCK_ROWS}×{FILTER_BLOCK_COLS}, sparsity {FILTER_SPARSITY}",
             fontsize=13, fontweight="bold")
ax.grid(True, linestyle="--", alpha=0.35)
ax.legend(loc="best", frameon=False)
fig.tight_layout()
plt.show()

## 9. (Optional) Compare across block sizes

If you ran experiments with multiple block sizes, this picks **one algorithm** and sweeps block size.

In [ ]:
FOCUS_ALG = "our_tetris"

block_sizes = sorted(df[df.algorithm == FOCUS_ALG].block_size.unique(),
                     key=lambda s: (int(s.split("x")[0]), int(s.split("x")[1])))

if len(block_sizes) > 1:
    fig, axes = plt.subplots(2, 4, figsize=(18, 8), sharex=True)
    axes_flat = axes.flatten()
    cmap = plt.cm.viridis(np.linspace(0, 0.85, len(block_sizes)))

    for i, lt in enumerate(LAYER_TYPES):
        ax = axes_flat[i]
        for bs, color in zip(block_sizes, cmap):
            sub = df[(df.algorithm == FOCUS_ALG) & (df.layer_type == lt) & (df.block_size == bs)]
            sub = sub.sort_values("layer_idx")
            ax.plot(sub["layer_idx"], sub["rel_error"],
                    label=f"block {bs}", color=color, linewidth=1.4, marker="o", markersize=3.5)
        ax.set_title(lt, fontsize=12, fontweight="bold")
        ax.grid(True, linestyle="--", alpha=0.35)
        if i % 4 == 0: ax.set_ylabel("Rel. error")
        if i >= 4:     ax.set_xlabel("Transformer block index")

    axes_flat[7].axis("off")
    handles = [Line2D([0], [0], color=c, lw=1.8, marker="o", markersize=4, label=f"block {bs}")
               for bs, c in zip(block_sizes, cmap)]
    axes_flat[7].legend(handles=handles, loc="center", fontsize=11, frameon=False,
                        title=f"{FOCUS_ALG}", title_fontsize=12)
    fig.suptitle(f"Block-size sweep for {FOCUS_ALG}", fontsize=14, fontweight="bold", y=1.00)
    fig.tight_layout()
    plt.show()
else:
    print(f"Only one block size ({block_sizes[0]}) in data for {FOCUS_ALG} — nothing to sweep.")